# 🚀 OpenMythos (Glasseye) on Google Colab

## Recurrent-Depth Transformer with GPU Inference

**Features:**
- ✅ Prelude → Recurrent Block → Coda architecture
- ✅ MLA/GQA attention modes
- ✅ Mixture of Experts (MoE) FFN
- ✅ Adaptive Computation Time (ACT)
- ✅ GPU-accelerated inference

**This notebook:**
1. Installs OpenMythos
2. Builds a tiny checkpoint
3. Runs inference
4. Optionally trains on FineWeb-Edu
5. Deploys FastAPI server

## 🔧 Setup & Installation

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone OpenMythos repository
!git clone https://github.com/MITRE-Cyber-Security-CVE-Database/ArtificialAutism.git
%cd ArtificialAutism

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets click pydantic loguru pyyaml fastapi uvicorn
!pip install -q -e .
print("✅ Installation complete!")

## 🏗️ Build Model Checkpoint

In [ ]:
# Build tiny checkpoint for testing
!python -m mythos_glasseye.cli build --size tiny --device cuda

## 🧪 Test Inference

In [ ]:
import torch
from open_mythos import OpenMythos, MythosConfig
from open_mythos.variants import mythos_1b

# Load checkpoint
checkpoint_path = "artifacts/openmythos_0ai_cpu_dev.pt"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model on {device}...")
checkpoint = torch.load(checkpoint_path, map_location=device)
config = checkpoint.get("config")
state_dict = checkpoint.get("state_dict")

model = OpenMythos(config)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print(f"✅ Model loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Generate text
input_ids = torch.randint(0, config.vocab_size, (1, 16)).to(device)

print("Generating...")
with torch.no_grad():
    output = model.generate(
        input_ids,
        max_new_tokens=20,
        n_loops=4
    )

print(f"✅ Generated {output.shape[1]} tokens")
print(f"Output shape: {output.shape}")
print(f"Output IDs: {output[0][:30].tolist()}")

## 🚂 Optional: Training

In [ ]:
# Prepare dataset
!python -m mythos_glasseye.cli prepare --source HuggingFaceFW/fineweb-edu --subset sample-10BT

In [ ]:
# Train (adjust steps for Colab time limits)
# Warning: Training uses GPU time - monitor your Colab usage
!python -m mythos_glasseye.cli train --steps 100 --batch-size 2

## 🚀 Deploy FastAPI Server

In [ ]:
# Generate FastAPI inference server
!python -m mythos_glasseye.cli serve --model artifacts/openmythos_0ai_cpu_dev.pt --mode fastapi --port 5000

In [ ]:
# Install pyngrok for public URL (optional)
!pip install -q pyngrok

from pyngrok import ngrok
import subprocess
import time

# Start FastAPI server in background
print("Starting FastAPI server...")
server_process = subprocess.Popen(
    ["uvicorn", "artifacts.inference_server:app", "--host", "0.0.0.0", "--port", "5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

# Create public URL with ngrok
public_url = ngrok.connect(5000)
print(f"\n✅ FastAPI server running!")
print(f"🌐 Public URL: {public_url}")
print(f"📚 Docs: {public_url}/docs")
print(f"❤️ Health: {public_url}/health")
print(f"🤖 Generate: {public_url}/generate")

In [ ]:
# Test the API
import requests
import json

# Health check
response = requests.get(f"{public_url}/health")
print("Health check:", response.json())

# Generate request
generate_request = {
    "input_ids": [1, 2, 3, 4, 5, 6, 7, 8],
    "max_new_tokens": 20,
    "n_loops": 4
}

response = requests.post(
    f"{public_url}/generate",
    json=generate_request
)

print("\nGeneration response:")
print(json.dumps(response.json(), indent=2))

## 📊 Monitor with TensorBoard

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir artifacts/runs

## 💾 Save & Download Models

In [ ]:
# Zip artifacts for download
!zip -r openmythos_artifacts.zip artifacts/

from google.colab import files
files.download('openmythos_artifacts.zip')
print("✅ Artifacts ready for download!")

## 🎯 Next Steps

1. **Train longer**: Increase `--steps` in training
2. **Deploy production**: Use Replicate or Together AI
3. **Scale up**: Rent GPU on Vast.ai or RunPod
4. **Experiment**: Try different model sizes and configs

---

**Repository**: https://github.com/MITRE-Cyber-Security-CVE-Database/ArtificialAutism  
**Documentation**: See `mythos_glasseye/README.md`  
**License**: MIT